In [ ]:
!pip install setfit datasets scikit-learn pandas tabulate -q

import pandas as pd
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from setfit import SetFitModel, SetFitTrainer
from datasets import Dataset
from tabulate import tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
# ======================
# 1. Upload file CSV manual di Colab
# ======================
from google.colab import files
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)

print(df.head())

Saving hasil_prediksi_sentimen - hasil_prediksi_sentimen (1).csv to hasil_prediksi_sentimen - hasil_prediksi_sentimen (1).csv
                                                text                span  \
0                   kenapa ya suka berobat ke penang             general   
1  kalau terduga maling jelaslah aneh berobat ke ...             general   
2  benar apa yg kiki katakan kebanyakan dokter di...           pelayanan   
3  benar apa yg kiki katakan kebanyakan dokter di...       aksesibilitas   
4  benar apa yg kiki katakan kebanyakan dokter di...  kualitas_diagnosis   

     label  ordinal  
0  positif      0.0  
1  negatif      0.0  
2  negatif      0.0  
3  positif      0.0  
4  positif      0.0  


In [ ]:
df

,text,span,label,ordinal
0,kenapa ya suka berobat ke penang,general,positif,0.0
1,kalau terduga maling jelaslah aneh berobat ke ...,general,negatif,0.0
2,benar apa yg kiki katakan kebanyakan dokter di...,pelayanan,negatif,0.0
3,benar apa yg kiki katakan kebanyakan dokter di...,aksesibilitas,positif,0.0
4,benar apa yg kiki katakan kebanyakan dokter di...,kualitas_diagnosis,positif,0.0
...,...,...,...,...
2666,orang keturunan china itu yg suka berobat ke l...,biaya,negatif,0.0
2667,orang keturunan china itu yg suka berobat ke l...,aksesibilitas,negatif,0.0
2668,maka dari itu pak presiden nyingung mentri nya...,general,positif,0.0
2669,apa ada kontak yg bisa dihubungi untuk informa...,aksesibilitas,positif,0.0


In [ ]:
from sklearn.model_selection import train_test_split

def split_and_sample(df_aspect):
    # Hanya split 80% train, 20% test tanpa batasi jumlah data
    train_df, test_df = train_test_split(df_aspect, test_size=0.2, random_state=42)
    return train_df, test_df


# ======================
# 2. Loop training per aspek
# ======================
aspect_names = df["span"].unique()
results = []

for aspect in aspect_names:
    print(f"\n=== Training untuk aspek: {aspect} ===")

    df_aspect = df[df["span"] == aspect]

    # Panggil tanpa batasan
    train_df, test_df = split_and_sample(df_aspect)

    if len(train_df) < 2 or len(test_df) < 1:
        print(f"Skip aspek {aspect} karena data terlalu sedikit")
        continue


    # Konversi ke HuggingFace Dataset
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)

    # Load model ringan biar hemat GPU
    model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-MiniLM-L6-v2")

    # Trainer
    trainer = SetFitTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        batch_size=16,           # lebih besar untuk stabilitas
        num_iterations=30,       # lebih banyak pasangan per aspek
        num_epochs=5             # training lebih lama
    )

    # Training & ukur waktu
    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time

    # Evaluasi
    y_true = test_dataset["label"]
    y_pred = trainer.model.predict(test_dataset["text"])

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    results.append([aspect, elapsed, acc, prec, rec, f1])

# ======================
# 3. Tampilkan hasil tabel
# ======================
headers = ["Aspect", "Time(s)", "Accuracy", "Precision", "Recall", "F1-score"]
print("\n===== HASIL AKURASI PER ASPEK =====")
print(tabulate(results, headers=headers, floatfmt=".4f"))



=== Training untuk aspek: general ===


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

Map:   0%|          | 0/410 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 24600
  Batch size = 16
  Num epochs = 5


Step,Training Loss
1,0.236500
50,0.235900
100,0.225200
150,0.215700
200,0.203500
250,0.146700
300,0.103800
350,0.081500
400,0.064500
450,0.044400



===== HASIL AKURASI PER ASPEK =====
Aspect                Time(s)    Accuracy    Precision    Recall    F1-score
------------------  ---------  ----------  -----------  --------  ----------
general              333.6288      0.6809       0.6847    0.6809      0.6800
pelayanan           1287.4170      0.9291       0.9186    0.9291      0.9124
aksesibilitas        842.8252      0.7253       0.7371    0.7253      0.7200
kualitas_diagnosis   597.3786      0.8906       0.9030    0.8906      0.8591
fasilitas            844.0860      0.8791       0.8357    0.8791      0.8491
biaya                946.3798      0.9515       0.9429    0.9515      0.9437


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Ambil 300 data untuk train
train_df = df.sample(n=300, random_state=42)
predict_df = df.drop(train_df.index)

# Encode label jadi angka
le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["label"])

# Konversi ke HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df)

# Load model
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-MiniLM-L6-v2")

# Trainer
trainer = SetFitTrainer(
    model=model,
    train_dataset=train_dataset,
    batch_size=16,
    num_iterations=30,
    num_epochs=5
)

# Train
trainer.train()

# Prediksi sisa data
y_pred = trainer.model.predict(predict_df["text"])

# Jika mau kembalikan label asli
y_pred_labels = le.inverse_transform(y_pred)

# Gabungkan hasil prediksi
predict_df = predict_df.assign(pred_label=y_pred_labels)

print(predict_df.head())


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
/tmp/ipython-input-1741783565.py:18: DeprecationWarning: `SetFitTrainer` has been deprecated and will be removed in v2.0.0 of SetFit. Please use `Trainer` instead.
  trainer = SetFitTrainer(


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 18000
  Batch size = 16
  Num epochs = 5


Step,Training Loss
1,0.250200
50,0.286100
100,0.261300
150,0.241700
200,0.223800
250,0.194200
300,0.152800
350,0.079300
400,0.032700
450,0.021400


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(


                                                text                span  \
0  jika orang yg memilih berobat ke luar negeri m...               biaya   
1  jika orang yg memilih berobat ke luar negeri m...           fasilitas   
2  tahun yang lalu adik kandung saya pernah di di...           pelayanan   
3  tahun yang lalu adik kandung saya pernah di di...       aksesibilitas   
4  tahun yang lalu adik kandung saya pernah di di...  kualitas_diagnosis   

     label  ordinal pred_label  
0  negatif      0.0        NaN  
1  negatif      0.0        NaN  
2  negatif      0.0        NaN  
3  negatif      0.0        NaN  
4  negatif      0.0        NaN  


In [ ]:
import setfit
print(setfit.__version__)


1.1.3


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
from datasets import Dataset
from setfit import SetFitModel, Trainer

# Ambil sample 300 data secara random
train_df = df.sample(n=300, random_state=42)
predict_df = df.drop(train_df.index)

# Encode label jadi angka
le = LabelEncoder()
train_df["label_encoded"] = le.fit_transform(train_df["label"])

# Pisah kelas mayoritas dan minoritas
majority_class = train_df["label_encoded"].value_counts().idxmax()
df_majority = train_df[train_df["label_encoded"] == majority_class]
df_minority = train_df[train_df["label_encoded"] != majority_class]

# Oversampling minoritas supaya seimbang dengan mayoritas
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

# Gabungkan kembali jadi balanced dataframe
train_df_balanced = pd.concat([df_majority, df_minority_upsampled])

# Shuffle data supaya acak
train_df_balanced = train_df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Buat dataset HuggingFace dari dataframe balanced
train_dataset = Dataset.from_pandas(
    train_df_balanced[["text", "label_encoded"]].rename(columns={"label_encoded": "label"})
)

# Load model SetFit
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-MiniLM-L6-v2")

# Inisiasi Trainer
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
)

# Train dengan parameter di method train()
trainer.train(
    num_iterations=30,
    num_epochs=5,
    batch_size=16,
)

# Prediksi data yang belum berlabel
y_pred = model.predict(predict_df["text"].tolist())

# Kembalikan label asli
y_pred_labels = le.inverse_transform(y_pred)

# Gabungkan hasil prediksi ke dataframe
predict_df = predict_df.assign(pred_label=y_pred_labels)

print(predict_df.head())
print(predict_df["pred_label"].value_counts())



model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Map:   0%|          | 0/530 [00:00<?, ? examples/s]

/tmp/ipython-input-716142059.py:49: DeprecationWarning: `Trainer.train` does not accept keyword arguments anymore. Please provide training arguments via a `TrainingArguments` instance to the `Trainer` initialisation or the `Trainer.train` method.
  trainer.train(
***** Running training *****
  Num unique pairs = 163550
  Batch size = 16
  Num epochs = 1


Step,Training Loss
1,0.216600
50,0.273200
100,0.265700
150,0.257600
200,0.251700
250,0.246400
300,0.232200
350,0.234600
400,0.218700
450,0.205800


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(


                                                text                span  \
0  jika orang yg memilih berobat ke luar negeri m...               biaya   
1  jika orang yg memilih berobat ke luar negeri m...           fasilitas   
2  tahun yang lalu adik kandung saya pernah di di...           pelayanan   
3  tahun yang lalu adik kandung saya pernah di di...       aksesibilitas   
4  tahun yang lalu adik kandung saya pernah di di...  kualitas_diagnosis   

     label  ordinal pred_label  
0  negatif      0.0        NaN  
1  negatif      0.0        NaN  
2  negatif      0.0        NaN  
3  negatif      0.0        NaN  
4  negatif      0.0        NaN  
pred_label
negatif    81
positif    17
Name: count, dtype: int64


In [ ]:
# Simpan hasil prediksi ke Excel
predict_df.to_excel("predictions.xlsx", index=False)

# Kalau di Google Colab, unduh
from google.colab import files
files.download("predictions.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Prediksi data yang belum berlabel dengan model.predict()
y_pred = model.predict(predict_df["text"].tolist())

# Kembalikan label asli
y_pred_labels = le.inverse_transform(y_pred)

# Gabungkan hasil prediksi ke dataframe
predict_df = predict_df.assign(pred_label=y_pred_labels)


In [ ]:
!pip install setfit datasets scikit-learn pandas -q

import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from setfit import SetFitModel, Trainer
from google.colab import files

In [ ]:
# Upload file CSV untuk data training berlabel
print("Upload file CSV data training (berlabel):")
uploaded_train = files.upload()
train_filename = list(uploaded_train.keys())[0]
train_df = pd.read_csv(train_filename)

Upload file CSV data training (berlabel):


Saving data 6 aspects explode (LABEL).csv to data 6 aspects explode (LABEL).csv


In [ ]:
# Upload file CSV untuk data yang belum berlabel
print("Upload file CSV data untuk prediksi (belum berlabel):")
uploaded_pred = files.upload()
predict_filename = list(uploaded_pred.keys())[0]
predict_df = pd.read_csv(predict_filename)

Upload file CSV data untuk prediksi (belum berlabel):


Saving data 6 aspects explode (BLM LABEL) - Copy.csv to data 6 aspects explode (BLM LABEL) - Copy.csv


In [ ]:
# --- Persiapan data training ---
train_df["input_text"] = train_df["span"] + ": " + train_df["text"]

le = LabelEncoder()
train_df["label_encoded"] = le.fit_transform(train_df["label"])

train_dataset = Dataset.from_pandas(
    train_df[["input_text", "label_encoded"]].rename(columns={"input_text": "text", "label_encoded": "label"})
)

# --- Load dan train model SetFit ---
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-MiniLM-L6-v2")
trainer = Trainer(model=model, train_dataset=train_dataset)
trainer.train(num_iterations=30, num_epochs=5, batch_size=16)

# --- Prediksi data uji ---
predict_df["input_text"] = predict_df["span"] + ": " + predict_df["text"]
y_pred = model.predict(predict_df["input_text"].tolist())
y_pred_labels = le.inverse_transform(y_pred)

predict_df = predict_df.assign(sentiment_pred=y_pred_labels)

print(predict_df.head())

# --- Simpan hasil prediksi ---
predict_df.to_csv("hasil_prediksi_sentimen.csv", index=False)

# --- Download hasil prediksi ---
files.download("hasil_prediksi_sentimen.csv")

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

/tmp/ipython-input-446430216.py:14: DeprecationWarning: `Trainer.train` does not accept keyword arguments anymore. Please provide training arguments via a `TrainingArguments` instance to the `Trainer` initialisation or the `Trainer.train` method.
  trainer.train(num_iterations=30, num_epochs=5, batch_size=16)
***** Running training *****
  Num unique pairs = 53582
  Batch size = 16
  Num epochs = 1


Step,Training Loss
1,0.252600
50,0.263900
100,0.255600
150,0.239800
200,0.227600
250,0.199800
300,0.180600
350,0.145000
400,0.113200
450,0.085200


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(


                                                text                span  \
0                   kenapa ya suka berobat ke penang             general   
1  kalau terduga maling jelaslah aneh berobat ke ...             general   
2  benar apa yg kiki katakan kebanyakan dokter di...           pelayanan   
3  benar apa yg kiki katakan kebanyakan dokter di...       aksesibilitas   
4  benar apa yg kiki katakan kebanyakan dokter di...  kualitas_diagnosis   

   label  ordinal                                         input_text  \
0    NaN      0.0          general: kenapa ya suka berobat ke penang   
1    NaN      0.0  general: kalau terduga maling jelaslah aneh be...   
2    NaN      0.0  pelayanan: benar apa yg kiki katakan kebanyaka...   
3    NaN      0.0  aksesibilitas: benar apa yg kiki katakan keban...   
4    NaN      0.0  kualitas_diagnosis: benar apa yg kiki katakan ...   

  sentiment_pred  
0        positif  
1        negatif  
2        negatif  
3        positif  
4        positi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Simpan hasil prediksi ke CSV
predict_df.to_csv("hasil_prediksi.csv", index=False)

# Download file CSV ke komputer lokal
from google.colab import files
files.download("hasil_prediksi.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>